# Ablation Study: Pattern-Driven Code Optimization Benchmark

This notebook runs a comprehensive ablation study across:
1. **Prompting strategies**: generic vs pattern-aware vs taxonomy-guided
2. **LoRA fine-tuning**: base Qwen2.5-Coder-7B vs 3 LoRA adapters
3. **Pattern categories**: which categories are easiest/hardest for LLMs
4. **Difficulty levels**: easy vs medium vs hard
5. **Model families**: closed-source (Claude, GPT, Gemini) vs open-weight vs fine-tuned

In [ ]:
import os, sys, json, csv, time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
from pathlib import Path
from collections import defaultdict
from IPython.display import display, Markdown

matplotlib.rcParams.update({"font.size": 12, "figure.figsize": (14, 6)})
sns.set_theme(style="whitegrid")

# Add project root to path
PROJECT_ROOT = Path(".").resolve()
sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")

In [ ]:
import os, sys, json, csv, time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
from pathlib import Path
from collections import defaultdict
from IPython.display import display, Markdown

matplotlib.rcParams.update({"font.size": 12, "figure.figsize": (14, 6)})
sns.set_theme(style="whitegrid")

# Add project root to path
PROJECT_ROOT = Path(".").resolve()
sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")

## 0. Dataset Overview

In [ ]:
# Load all variant metadata
metadata = []
for root, dirs, files in os.walk("dataset"):
    if "metadata.json" in files:
        with open(os.path.join(root, "metadata.json")) as f:
            metadata.append(json.load(f))

df_meta = pd.DataFrame(metadata)
print(f"Total variants: {len(df_meta)}")
print(f"Unique patterns: {df_meta['pattern_id'].nunique()}")
print(f"Categories: {df_meta['category'].nunique()}")
print()

# Summary table
summary = df_meta.groupby("category").agg(
    patterns=("pattern_id", "nunique"),
    variants=("variant_id", "count"),
    difficulties=("difficulty", lambda x: dict(x.value_counts()))
).sort_values("variants", ascending=False)
display(summary)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Variants per category
cat_counts = df_meta["category"].value_counts().sort_values()
cat_counts.plot.barh(ax=axes[0], color=sns.color_palette("viridis", len(cat_counts)))
axes[0].set_title("Variants per Category")
axes[0].set_xlabel("Count")

# Difficulty distribution
diff_counts = df_meta["difficulty"].value_counts()
diff_counts.plot.pie(ax=axes[1], autopct="%1.0f%%", colors=sns.color_palette("Set2", 3))
axes[1].set_title("Difficulty Distribution")
axes[1].set_ylabel("")

plt.tight_layout()
plt.savefig("figures/dataset_overview.png", dpi=150, bbox_inches="tight")
plt.show()

## 1. Run Evaluation (or Load Cached Results)

This section either runs the full evaluation or loads previously saved results.

**To run fresh evaluations**, set `RUN_EVAL = True` and configure which models to evaluate.

**To load cached results**, place your `results.csv` in the project root.

In [ ]:
RUN_EVAL = False  # Set True to run evaluations (requires API keys / GPU)
RESULTS_CSV = "results.csv"

# Models to evaluate — uncomment the ones you want
EVAL_CONFIGS = [
    # ── LoRA fine-tuned (local GPU) ──
    {"model": "qwen-lora-generic",   "strategy": "generic"},
    {"model": "qwen-lora-pattern",   "strategy": "pattern-aware"},
    {"model": "qwen-lora-taxonomy",  "strategy": "taxonomy-guided"},

    # ── Base model baseline (local via Ollama) ──
    {"model": "qwen2.5-coder-7b-ollama", "strategy": "generic"},
    {"model": "qwen2.5-coder-7b-ollama", "strategy": "pattern-aware"},
    {"model": "qwen2.5-coder-7b-ollama", "strategy": "taxonomy-guided"},

    # ── Closed-source (requires API keys) ──
    # {"model": "claude-sonnet-4-6",  "strategy": "generic"},
    # {"model": "claude-sonnet-4-6",  "strategy": "pattern-aware"},
    # {"model": "claude-sonnet-4-6",  "strategy": "taxonomy-guided"},
    # {"model": "gpt-4o",            "strategy": "generic"},
    # {"model": "gpt-4o",            "strategy": "pattern-aware"},
    # {"model": "gpt-4o",            "strategy": "taxonomy-guided"},
    # {"model": "gemini-2-flash",    "strategy": "generic"},
    # {"model": "deepseek-v3",       "strategy": "generic"},
]

In [ ]:
if RUN_EVAL:
    from models import load_model_config, make_call_llm_fn
    from evaluator import evaluate_pattern
    from patterns import PATTERNS
    from dataclasses import asdict

    models_cfg = load_model_config("models.yaml")
    active_patterns = [p for p in PATTERNS if p.test_harness.strip()]

    all_results = []
    for cfg in EVAL_CONFIGS:
        model_id, strategy = cfg["model"], cfg["strategy"]
        print(f"\n{'='*60}")
        print(f"Model: {model_id}  |  Strategy: {strategy}")
        print(f"{'='*60}")

        mcfg = models_cfg[model_id]
        call_llm_fn = make_call_llm_fn(mcfg, retries=2)

        for i, pattern in enumerate(active_patterns):
            print(f"  [{i+1}/{len(active_patterns)}] {pattern.pattern_id:<8}", end="", flush=True)
            t0 = time.time()
            try:
                result = evaluate_pattern(pattern, model_id, strategy, call_llm_fn)
                elapsed = time.time() - t0
                status = "OK" if result.compiles and result.correct else "FAIL"
                print(f" {status} ({elapsed:.1f}s) speedup={result.speedup_vs_slow:.1f}x")
                all_results.append(asdict(result))
            except Exception as e:
                print(f" ERROR: {e}")

    # Save results
    df_results = pd.DataFrame(all_results)
    df_results.to_csv(RESULTS_CSV, index=False)
    print(f"\nSaved {len(df_results)} results to {RESULTS_CSV}")
else:
    if os.path.exists(RESULTS_CSV):
        df_results = pd.read_csv(RESULTS_CSV)
        print(f"Loaded {len(df_results)} results from {RESULTS_CSV}")
        print(f"Models: {df_results['model'].unique()}")
        print(f"Strategies: {df_results['strategy'].unique()}")
    else:
        print(f"No results file found at {RESULTS_CSV}.")
        print("Set RUN_EVAL = False and configure EVAL_CONFIGS to run evaluations.")
        df_results = pd.DataFrame()

In [ ]:
# Merge metadata with results for richer analysis
if not df_results.empty:
    df = df_results.merge(
        df_meta[["pattern_id", "category", "difficulty", "compiler_fixable"]].drop_duplicates("pattern_id"),
        on="pattern_id", how="left"
    )
    print(f"Merged dataset: {len(df)} rows")
    display(df.head())
else:
    df = pd.DataFrame()
    print("No results to analyze. Run evaluation first.")

## 2. Ablation 1: Prompting Strategy

Compare **generic** vs **pattern-aware** vs **taxonomy-guided** prompting across all models.

In [ ]:
if not df.empty:
    # Compile rate and correctness by strategy
    strat_summary = df.groupby("strategy").agg(
        n=("pattern_id", "count"),
        compile_rate=("compiles", "mean"),
        correct_rate=("correct", "mean"),
        mean_speedup=("speedup_vs_slow", lambda x: x[df.loc[x.index, "correct"]].mean()),
        median_speedup=("speedup_vs_slow", lambda x: x[df.loc[x.index, "correct"]].median()),
    ).round(3)
    display(Markdown("### Overall Strategy Comparison"))
    display(strat_summary)

    # Per-model strategy comparison
    pivot = df[df["correct"]].groupby(["model", "strategy"])["speedup_vs_slow"].agg(["mean", "median", "count"]).round(2)
    display(Markdown("### Per-Model Strategy Breakdown"))
    display(pivot)

In [ ]:
if not df.empty and df["strategy"].nunique() > 1:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # Compile rate by strategy
    df.groupby("strategy")["compiles"].mean().plot.bar(ax=axes[0], color=sns.color_palette("Set2", 3))
    axes[0].set_title("Compile Rate by Strategy")
    axes[0].set_ylabel("Rate")
    axes[0].set_ylim(0, 1.05)
    axes[0].tick_params(axis="x", rotation=15)

    # Correctness rate by strategy
    df.groupby("strategy")["correct"].mean().plot.bar(ax=axes[1], color=sns.color_palette("Set2", 3))
    axes[1].set_title("Correctness Rate by Strategy")
    axes[1].set_ylabel("Rate")
    axes[1].set_ylim(0, 1.05)
    axes[1].tick_params(axis="x", rotation=15)

    # Speedup distribution by strategy
    correct_df = df[df["correct"]]
    if not correct_df.empty:
        sns.boxplot(data=correct_df, x="strategy", y="speedup_vs_slow", ax=axes[2], palette="Set2")
        axes[2].set_title("Speedup vs Slow (correct only)")
        axes[2].set_ylabel("Speedup")
        axes[2].set_yscale("log")
        axes[2].tick_params(axis="x", rotation=15)

    plt.tight_layout()
    plt.savefig("figures/ablation_strategy.png", dpi=150, bbox_inches="tight")
    plt.show()

## 3. Ablation 2: LoRA Fine-Tuning Effect

Compare base Qwen2.5-Coder-7B vs each LoRA adapter with its matched strategy.

In [ ]:
if not df.empty:
    # Tag models as base or fine-tuned
    df["is_finetuned"] = df["model"].str.contains("lora")
    df["model_type"] = df["is_finetuned"].map({True: "LoRA Fine-tuned", False: "Base / API"})

    lora_comparison = df.groupby(["model", "model_type", "strategy"]).agg(
        compile_rate=("compiles", "mean"),
        correct_rate=("correct", "mean"),
        mean_speedup=("speedup_vs_slow", lambda x: x[df.loc[x.index, "correct"]].mean() if df.loc[x.index, "correct"].any() else 0),
    ).round(3)

    display(Markdown("### Base vs Fine-Tuned Comparison"))
    display(lora_comparison)

In [ ]:
if not df.empty and df["is_finetuned"].any():
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Correctness: base vs fine-tuned per strategy
    ct = df.groupby(["strategy", "model_type"])["correct"].mean().unstack()
    ct.plot.bar(ax=axes[0], color=["#66c2a5", "#fc8d62"])
    axes[0].set_title("Correctness Rate: Base vs LoRA")
    axes[0].set_ylabel("Correct Rate")
    axes[0].set_ylim(0, 1.05)
    axes[0].legend(title="Model Type")
    axes[0].tick_params(axis="x", rotation=15)

    # Speedup: base vs fine-tuned
    correct_df = df[df["correct"]]
    if not correct_df.empty:
        sp = correct_df.groupby(["strategy", "model_type"])["speedup_vs_slow"].median().unstack()
        sp.plot.bar(ax=axes[1], color=["#66c2a5", "#fc8d62"])
        axes[1].set_title("Median Speedup: Base vs LoRA (correct only)")
        axes[1].set_ylabel("Median Speedup vs Slow")
        axes[1].legend(title="Model Type")
        axes[1].tick_params(axis="x", rotation=15)

    plt.tight_layout()
    plt.savefig("figures/ablation_lora.png", dpi=150, bbox_inches="tight")
    plt.show()

## 4. Ablation 3: Pattern Category Analysis

Which optimization categories are easiest/hardest for LLMs?

In [ ]:
if not df.empty:
    cat_perf = df.groupby("category").agg(
        n=("pattern_id", "count"),
        compile_rate=("compiles", "mean"),
        correct_rate=("correct", "mean"),
        mean_speedup=("speedup_vs_slow", lambda x: x[df.loc[x.index, "correct"]].mean() if df.loc[x.index, "correct"].any() else 0),
    ).round(3).sort_values("correct_rate", ascending=False)

    display(Markdown("### Performance by Category (all models pooled)"))
    display(cat_perf)

In [ ]:
if not df.empty:
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    # Correctness by category
    cat_correct = df.groupby("category")["correct"].mean().sort_values()
    cat_correct.plot.barh(ax=axes[0], color=sns.color_palette("RdYlGn", len(cat_correct)))
    axes[0].set_title("Correctness Rate by Category")
    axes[0].set_xlabel("Correct Rate")
    axes[0].set_xlim(0, 1.05)

    # Speedup by category (correct only)
    correct_df = df[df["correct"]]
    if not correct_df.empty:
        cat_speed = correct_df.groupby("category")["speedup_vs_slow"].median().sort_values()
        cat_speed.plot.barh(ax=axes[1], color=sns.color_palette("coolwarm", len(cat_speed)))
        axes[1].set_title("Median Speedup by Category (correct only)")
        axes[1].set_xlabel("Median Speedup vs Slow")
        axes[1].set_xscale("log")

    plt.tight_layout()
    plt.savefig("figures/ablation_category.png", dpi=150, bbox_inches="tight")
    plt.show()

In [ ]:
if not df.empty and df["model"].nunique() > 1:
    # Heatmap: correctness by model x category
    pivot_correct = df.pivot_table(
        index="model", columns="category", values="correct", aggfunc="mean"
    ).round(2)

    fig, ax = plt.subplots(figsize=(16, max(4, len(pivot_correct) * 0.6)))
    sns.heatmap(pivot_correct, annot=True, fmt=".2f", cmap="RdYlGn",
                vmin=0, vmax=1, ax=ax, linewidths=0.5)
    ax.set_title("Correctness Rate: Model × Category")
    plt.tight_layout()
    plt.savefig("figures/heatmap_model_category.png", dpi=150, bbox_inches="tight")
    plt.show()

## 5. Ablation 4: Difficulty Analysis

In [ ]:
if not df.empty:
    diff_perf = df.groupby("difficulty").agg(
        n=("pattern_id", "count"),
        compile_rate=("compiles", "mean"),
        correct_rate=("correct", "mean"),
        mean_speedup=("speedup_vs_slow", lambda x: x[df.loc[x.index, "correct"]].mean() if df.loc[x.index, "correct"].any() else 0),
    ).round(3)

    display(Markdown("### Performance by Difficulty"))
    display(diff_perf)

In [ ]:
if not df.empty:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    order = ["easy", "medium", "hard"]
    existing = [d for d in order if d in df["difficulty"].values]

    # Correctness by difficulty
    diff_correct = df.groupby("difficulty")["correct"].mean().reindex(existing)
    diff_correct.plot.bar(ax=axes[0], color=["#2ecc71", "#f39c12", "#e74c3c"][:len(existing)])
    axes[0].set_title("Correctness Rate by Difficulty")
    axes[0].set_ylabel("Correct Rate")
    axes[0].set_ylim(0, 1.05)
    axes[0].tick_params(axis="x", rotation=0)

    # Speedup by difficulty
    correct_df = df[df["correct"]]
    if not correct_df.empty:
        sns.boxplot(data=correct_df, x="difficulty", y="speedup_vs_slow",
                    order=existing, ax=axes[1], palette=["#2ecc71", "#f39c12", "#e74c3c"][:len(existing)])
        axes[1].set_title("Speedup Distribution by Difficulty")
        axes[1].set_yscale("log")

    plt.tight_layout()
    plt.savefig("figures/ablation_difficulty.png", dpi=150, bbox_inches="tight")
    plt.show()

## 6. Ablation 5: Model Family Comparison

In [ ]:
if not df.empty:
    # Classify models into families
    def get_model_family(model_id):
        if "lora" in model_id:
            return "Qwen2.5-Coder + LoRA"
        if "claude" in model_id:
            return "Claude"
        if "gpt" in model_id or "o3" in model_id:
            return "OpenAI"
        if "gemini" in model_id:
            return "Gemini"
        if "deepseek" in model_id:
            return "DeepSeek"
        if "llama" in model_id:
            return "Llama"
        if "qwen" in model_id:
            return "Qwen (base)"
        if "mistral" in model_id or "mixtral" in model_id:
            return "Mistral"
        if "gemma" in model_id:
            return "Gemma"
        if "phi" in model_id:
            return "Phi"
        if "falcon" in model_id:
            return "Falcon"
        return "Other"

    df["family"] = df["model"].apply(get_model_family)

    family_perf = df.groupby("family").agg(
        models=("model", "nunique"),
        n=("pattern_id", "count"),
        compile_rate=("compiles", "mean"),
        correct_rate=("correct", "mean"),
        mean_speedup=("speedup_vs_slow", lambda x: x[df.loc[x.index, "correct"]].mean() if df.loc[x.index, "correct"].any() else 0),
    ).round(3).sort_values("correct_rate", ascending=False)

    display(Markdown("### Performance by Model Family"))
    display(family_perf)

In [ ]:
if not df.empty and df["family"].nunique() > 1:
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    # Correctness by family
    fam_correct = df.groupby("family")["correct"].mean().sort_values()
    fam_correct.plot.barh(ax=axes[0], color=sns.color_palette("tab10", len(fam_correct)))
    axes[0].set_title("Correctness Rate by Model Family")
    axes[0].set_xlabel("Correct Rate")
    axes[0].set_xlim(0, 1.05)

    # Speedup by family
    correct_df = df[df["correct"]]
    if not correct_df.empty:
        fam_speed = correct_df.groupby("family")["speedup_vs_slow"].median().sort_values()
        fam_speed.plot.barh(ax=axes[1], color=sns.color_palette("tab10", len(fam_speed)))
        axes[1].set_title("Median Speedup by Model Family (correct only)")
        axes[1].set_xlabel("Median Speedup")
        axes[1].set_xscale("log")

    plt.tight_layout()
    plt.savefig("figures/ablation_model_family.png", dpi=150, bbox_inches="tight")
    plt.show()

## 7. Ablation 6: Speedup vs Reference (LLM vs Hand-Optimized)

In [ ]:
if not df.empty and "speedup_vs_ref" in df.columns:
    correct_df = df[df["correct"] & (df["speedup_vs_ref"] > 0)]
    if not correct_df.empty:
        display(Markdown("### LLM Output vs Hand-Optimized Reference"))
        display(Markdown(
            "`speedup_vs_ref > 1` means LLM is **faster** than hand-optimized.\n\n"
            "`speedup_vs_ref < 1` means LLM is **slower** than hand-optimized."
        ))

        ref_summary = correct_df.groupby("model")["speedup_vs_ref"].agg(
            ["mean", "median", "count",
             lambda x: (x >= 1.0).mean()]  # fraction matching or beating reference
        ).round(3)
        ref_summary.columns = ["mean", "median", "count", "pct_beats_ref"]
        display(ref_summary.sort_values("pct_beats_ref", ascending=False))

        # Distribution
        fig, ax = plt.subplots(figsize=(12, 5))
        for model in correct_df["model"].unique():
            subset = correct_df[correct_df["model"] == model]["speedup_vs_ref"]
            ax.hist(subset.clip(0, 5), bins=30, alpha=0.5, label=model)
        ax.axvline(1.0, color="red", linestyle="--", label="Reference (1x)")
        ax.set_xlabel("Speedup vs Hand-Optimized Reference")
        ax.set_ylabel("Count")
        ax.set_title("LLM vs Hand-Optimized: Speedup Distribution")
        ax.legend()
        plt.tight_layout()
        plt.savefig("figures/ablation_vs_reference.png", dpi=150, bbox_inches="tight")
        plt.show()

## 8. AST Faithfulness Analysis

Check if LLM outputs apply the *correct structural transformation* (not just any speedup).

In [ ]:
if not df.empty:
    try:
        sys.path.insert(0, str(PROJECT_ROOT / "faithfulness"))
        from checkers import check_faithfulness, CHECKERS

        # Run faithfulness check on correct LLM outputs
        correct_df = df[df["correct"]].copy()
        faith_results = []

        for _, row in correct_df.iterrows():
            pid = row["pattern_id"]
            if pid not in CHECKERS:
                continue
            # Load slow.c for this pattern
            pat_dir = pid.replace("-", "_")
            slow_path = PROJECT_ROOT / "dataset" / pat_dir / f"{pid}_v000" / "slow.c"
            if not slow_path.exists():
                continue
            slow_code = slow_path.read_text()
            llm_code = row.get("llm_code", "")
            if not llm_code:
                continue

            result = check_faithfulness(pid, slow_code, llm_code)
            faith_results.append({
                "pattern_id": pid,
                "model": row["model"],
                "strategy": row["strategy"],
                "verdict": result.verdict,
                "score": result.score,
            })

        if faith_results:
            df_faith = pd.DataFrame(faith_results)
            display(Markdown("### Faithfulness by Model"))
            faith_summary = df_faith.groupby("model").agg(
                n=("pattern_id", "count"),
                faithful_rate=("verdict", lambda x: (x == "faithful").mean()),
                avg_score=("score", "mean"),
            ).round(3)
            display(faith_summary.sort_values("faithful_rate", ascending=False))

            display(Markdown("### Faithfulness by Strategy"))
            display(df_faith.groupby("strategy")["verdict"].value_counts().unstack(fill_value=0))
        else:
            print("No faithfulness results (llm_code column may be empty in loaded CSV).")
    except ImportError as e:
        print(f"Faithfulness checker not available: {e}")
        print("Install pycparser: pip install pycparser")

## 9. Summary Table

In [ ]:
if not df.empty:
    display(Markdown("### Final Summary: All Models × Strategies"))

    final = df.groupby(["model", "strategy"]).agg(
        patterns=("pattern_id", "nunique"),
        compile_pct=("compiles", lambda x: f"{x.mean()*100:.1f}%"),
        correct_pct=("correct", lambda x: f"{x.mean()*100:.1f}%"),
        mean_speedup=("speedup_vs_slow", lambda x: round(x[df.loc[x.index, "correct"]].mean(), 1) if df.loc[x.index, "correct"].any() else 0),
        median_speedup=("speedup_vs_slow", lambda x: round(x[df.loc[x.index, "correct"]].median(), 1) if df.loc[x.index, "correct"].any() else 0),
    )
    display(final)

    # Save as LaTeX table for paper
    latex = final.to_latex()
    with open("figures/summary_table.tex", "w") as f:
        f.write(latex)
    print("\nLaTeX table saved to figures/summary_table.tex")

In [ ]:
# Create figures directory if it doesn't exist
os.makedirs("figures", exist_ok=True)
print("All figures saved to figures/")